# MetrôBot SP 2.0 — Solução completa

Notebook preparado para o desafio **MetrôBot SP 2.0**.

### O que está implementado
- Grafo com as 3 linhas e as 52 estações distintas.
- Estações de integração deduzidas por inferência.
- Regras R1–R7, incluindo a regra R6 de integração e uma nova regra R7.
- Busca de rotas com BFS e DFS.
- Interpretação de pedidos em linguagem natural em modo offline.
- Suporte opcional a um LLM compatível com API OpenAI/Groq, sem chave gravada no código.
- Narrador determinístico da rota.
- Interface com `ipywidgets`.
- Mapa visual das 3 linhas.
- Tabela-verdade.
- Testes automatizados com mais de 8 `assert`s.
- Os 6 casos obrigatórios do enunciado.

In [ ]:
from collections import deque
from itertools import product
from IPython.display import display, HTML, clear_output
import html
import json
import re
import unicodedata

try:
    import ipywidgets as widgets
except ImportError:
    !pip -q install ipywidgets
    import ipywidgets as widgets

print("Ambiente pronto.")

## 1. Dados das linhas

As estações abaixo seguem a organização do enunciado. Uma estação compartilhada continua sendo **um único nó do grafo**, mesmo quando pertence a duas linhas.

In [ ]:
LINHAS = {
    "Linha 1-Azul": [
        "Tucuruvi", "Parada Inglesa", "Jardim São Paulo", "Santana",
        "Carandiru", "Portuguesa-Tietê", "Armênia", "Tiradentes", "Luz",
        "São Bento", "Sé", "Japão-Liberdade", "São Joaquim", "Vergueiro",
        "Paraíso", "Ana Rosa", "Vila Mariana", "Santa Cruz",
        "Praça da Árvore", "Saúde", "São Judas", "Conceição", "Jabaquara"
    ],
    "Linha 2-Verde": [
        "Vila Madalena", "Sumaré", "Clínicas", "Consolação", "Trianon-Masp",
        "Brigadeiro", "Paraíso", "Ana Rosa", "Chácara Klabin",
        "Santos-Imigrantes", "Alto do Ipiranga", "Sacomã",
        "Tamanduateí", "Vila Prudente"
    ],
    "Linha 3-Vermelha": [
        "Palmeiras-Barra Funda", "Marechal Deodoro", "Santa Cecília",
        "República", "Anhangabaú", "Sé", "Pedro II", "Brás",
        "Bresser-Mooca", "Belém", "Tatuapé", "Carrão", "Penha",
        "Vila Matilde", "Guilhermina-Esperança", "Patriarca-Vila Ré",
        "Artur Alvim", "Corinthians-Itaquera"
    ]
}

CORES = {
    "Linha 1-Azul": "#1e88e5",
    "Linha 2-Verde": "#2e7d32",
    "Linha 3-Vermelha": "#d32f2f"
}

TODAS_ESTACOES = list(dict.fromkeys(e for linha in LINHAS.values() for e in linha))

print("Estações distintas:", len(TODAS_ESTACOES))
print("Estações:", ", ".join(TODAS_ESTACOES))

In [ ]:
def construir_grafo_multilinhas(linhas):
    grafo = {estacao: set() for lista in linhas.values() for estacao in lista}
    linhas_do_trecho = {}

    def registrar_aresta(a, b, linha):
        grafo[a].add(b)
        grafo[b].add(a)
        chave = tuple(sorted((a, b)))
        linhas_do_trecho.setdefault(chave, set()).add(linha)

    for linha, lista in linhas.items():
        for a, b in zip(lista, lista[1:]):
            registrar_aresta(a, b, linha)

    # Mantém a representação sem duplicação de vizinhos.
    grafo = {k: sorted(v) for k, v in grafo.items()}
    return grafo, linhas_do_trecho

GRAFO, LINHAS_DO_TRECHO = construir_grafo_multilinhas(LINHAS)

def chave_trecho(a, b):
    return tuple(sorted((a, b)))

def linhas_do_trecho(a, b):
    return sorted(LINHAS_DO_TRECHO.get(chave_trecho(a, b), set()))

def linhas_da_estacao(estacao):
    return [linha for linha, lista in LINHAS.items() if estacao in lista]

print("Nós do grafo:", len(GRAFO))
print("Exemplo de integração:")
for estacao in ["Sé", "Paraíso", "Ana Rosa"]:
    print(estacao, "->", linhas_da_estacao(estacao))

## 2. Inferência das integrações — R6

A regra pedida pelo enunciado é:

**∀e ∀l1 ∀l2 (pertence(e,l1) ∧ pertence(e,l2) ∧ l1 ≠ l2 → integracao(e))**

Portanto, não digitamos manualmente a lista de integrações. Ela é obtida verificando quais estações pertencem a pelo menos duas linhas.

In [ ]:
def inferir_integracoes():
    return sorted(
        estacao for estacao in GRAFO
        if len(linhas_da_estacao(estacao)) >= 2
    )

INTEGRACOES = inferir_integracoes()

print("Integrações inferidas:", INTEGRACOES)
assert set(INTEGRACOES) == {"Sé", "Paraíso", "Ana Rosa"}

## 3. Regras R1–R7

Para deixar o mecanismo verificável, as regras são representadas como fatos e inferências simples.

- **R1:** origem informada → fato `origem`.
- **R2:** destino informado → fato `destino`.
- **R3:** estação declarada fechada → fato `bloqueada`.
- **R4:** pedido de acessibilidade + elevador em manutenção → alerta de acessibilidade.
- **R5:** rota encontrada → fato `rota_encontrada`; caso contrário, `sem_rota`.
- **R6:** estação pertence a duas linhas diferentes → `integracao`.
- **R7:** nova regra útil: se houver estação fechada que pertença ao caminho normalmente pretendido, gerar aviso de desvio/indisponibilidade. No planejamento efetivo, isso aparece como regra de bloqueio que força a busca a procurar outro caminho.

In [ ]:
def motor_regras(origem, destino, bloqueadas=None, acessibilidade=False,
                 elevadores_manutencao=None, rota=None):
    bloqueadas = set(bloqueadas or [])
    elevadores_manutencao = set(elevadores_manutencao or [])
    rota = rota or []

    fatos = set()
    regras_disparadas = []

    # R1
    if origem in GRAFO:
        fatos.add(("origem", origem))
        regras_disparadas.append("R1: origem identificada.")

    # R2
    if destino in GRAFO:
        fatos.add(("destino", destino))
        regras_disparadas.append("R2: destino identificado.")

    # R3
    for estacao in bloqueadas:
        if estacao in GRAFO:
            fatos.add(("bloqueada", estacao))
    if bloqueadas:
        regras_disparadas.append(
            "R3: bloqueios considerados: " + ", ".join(sorted(bloqueadas)) + "."
        )

    # R4
    alertas_acessibilidade = []
    if acessibilidade:
        for estacao in rota:
            if estacao in elevadores_manutencao:
                alertas_acessibilidade.append(estacao)
        if alertas_acessibilidade:
            regras_disparadas.append(
                "R4: acessibilidade solicitada e há elevador em manutenção em "
                + ", ".join(alertas_acessibilidade) + "."
            )
        else:
            regras_disparadas.append(
                "R4: acessibilidade solicitada; nenhuma estação da rota "
                "está marcada com elevador em manutenção."
            )

    # R5
    if rota:
        fatos.add(("rota_encontrada", origem, destino))
        regras_disparadas.append("R5: rota encontrada.")
    else:
        fatos.add(("sem_rota", origem, destino))
        regras_disparadas.append("R5: nenhuma rota disponível com as restrições atuais.")

    # R6 — inferência automática
    for estacao in INTEGRACOES:
        fatos.add(("integracao", estacao))
    regras_disparadas.append(
        "R6: integrações inferidas automaticamente: " + ", ".join(INTEGRACOES) + "."
    )

    # R7 — nova regra útil
    if bloqueadas and rota:
        regras_disparadas.append(
            "R7: rota calculada considerando os bloqueios; o algoritmo evita "
            "estações fechadas e procura desvio quando existir."
        )
    elif bloqueadas and not rota:
        regras_disparadas.append(
            "R7: bloqueios ativos impediram a formação de uma rota."
        )
    else:
        regras_disparadas.append(
            "R7: nenhuma restrição adicional de desvio foi necessária."
        )

    return {
        "fatos": fatos,
        "regras_disparadas": regras_disparadas,
        "alertas_acessibilidade": alertas_acessibilidade
    }

## 4. Busca de rotas — BFS e DFS

A BFS encontra uma rota com menor número de trechos em um grafo não ponderado. A DFS também é disponibilizada para comparação, como solicitado na interface.

In [ ]:
def validar_estacao(estacao):
    return estacao in GRAFO

def bfs(origem, destino, bloqueadas=None):
    bloqueadas = set(bloqueadas or [])

    if origem not in GRAFO or destino not in GRAFO:
        return None, []
    if origem in bloqueadas or destino in bloqueadas:
        return None, []

    fila = deque([origem])
    anterior = {origem: None}
    visitados = []

    while fila:
        atual = fila.popleft()
        visitados.append(atual)

        if atual == destino:
            break

        for vizinho in GRAFO[atual]:
            if vizinho in bloqueadas or vizinho in anterior:
                continue
            anterior[vizinho] = atual
            fila.append(vizinho)

    if destino not in anterior:
        return None, visitados

    caminho = []
    atual = destino
    while atual is not None:
        caminho.append(atual)
        atual = anterior[atual]
    caminho.reverse()

    return caminho, visitados

def dfs(origem, destino, bloqueadas=None):
    bloqueadas = set(bloqueadas or [])

    if origem not in GRAFO or destino not in GRAFO:
        return None, []
    if origem in bloqueadas or destino in bloqueadas:
        return None, []

    pilha = [origem]
    anterior = {origem: None}
    visitados = []

    while pilha:
        atual = pilha.pop()
        if atual in visitados:
            continue

        visitados.append(atual)

        if atual == destino:
            break

        # Ordem reversa para manter uma exploração determinística.
        for vizinho in reversed(GRAFO[atual]):
            if vizinho in bloqueadas or vizinho in anterior:
                continue
            anterior[vizinho] = atual
            pilha.append(vizinho)

    if destino not in anterior:
        return None, visitados

    caminho = []
    atual = destino
    while atual is not None:
        caminho.append(atual)
        atual = anterior[atual]
    caminho.reverse()

    return caminho, visitados

## 5. Identificação de linha e baldeações

Uma aresta pode pertencer a duas linhas. O trecho **Paraíso–Ana Rosa**, por exemplo, pertence à Linha 1-Azul e à Linha 2-Verde.

Para contar transferências, o algoritmo tenta manter a linha atual quando ela continua disponível no próximo trecho. Assim, uma transferência é registrada somente quando é necessário trocar de linha.

In [ ]:
def analisar_linhas_e_transferencias(caminho):
    if not caminho or len(caminho) < 2:
        return [], []

    sequencia_linhas = []
    transferencias = []
    linha_atual = None

    for i in range(len(caminho) - 1):
        a, b = caminho[i], caminho[i + 1]
        candidatas = linhas_do_trecho(a, b)

        if not candidatas:
            continue

        if linha_atual in candidatas:
            escolhida = linha_atual
        else:
            escolhida = candidatas[0]
            if linha_atual is not None:
                transferencias.append({
                    "estacao": a,
                    "de": linha_atual,
                    "para": escolhida
                })

        sequencia_linhas.append(escolhida)
        linha_atual = escolhida

    return sequencia_linhas, transferencias

def contar_baldeacoes(caminho):
    _, transferencias = analisar_linhas_e_transferencias(caminho)
    return len(transferencias), [t["estacao"] for t in transferencias]

In [ ]:
def planejar_rota(origem, destino, bloqueadas=None, acessibilidade=False,
                  elevadores_manutencao=None, algoritmo="BFS"):
    bloqueadas = set(bloqueadas or [])
    elevadores_manutencao = set(elevadores_manutencao or [])

    if algoritmo.upper() == "DFS":
        caminho, visitados = dfs(origem, destino, bloqueadas)
    else:
        caminho, visitados = bfs(origem, destino, bloqueadas)

    if caminho:
        linhas, transferencias = analisar_linhas_e_transferencias(caminho)
        transferencias_nomes = [t["estacao"] for t in transferencias]
        tempo_estimado = max(0, (len(caminho) - 1) * 2)
    else:
        linhas, transferencias, transferencias_nomes = [], [], []
        tempo_estimado = None

    regras = motor_regras(
        origem, destino,
        bloqueadas=bloqueadas,
        acessibilidade=acessibilidade,
        elevadores_manutencao=elevadores_manutencao,
        rota=caminho or []
    )

    return {
        "origem": origem,
        "destino": destino,
        "bloqueadas": sorted(bloqueadas),
        "acessibilidade": acessibilidade,
        "elevadores_manutencao": sorted(elevadores_manutencao),
        "algoritmo": algoritmo.upper(),
        "rota": caminho,
        "visitados": visitados,
        "linhas": linhas,
        "transferencias": transferencias,
        "transferencias_nomes": transferencias_nomes,
        "paradas": len(caminho) - 1 if caminho else None,
        "tempo_estimado_min": tempo_estimado,
        "regras": regras
    }

## 6. Interpretador de linguagem natural

O modo offline reconhece as estações e alguns locais conhecidos. A saída é um JSON estruturado com origem, destino, acessibilidade, bloqueios e algoritmo.

Se quiser utilizar um LLM posteriormente, a função abaixo aceita uma configuração opcional via variável de ambiente, sem colocar chave de API no notebook.

In [ ]:
def normalizar(txt):
    txt = unicodedata.normalize("NFKD", txt)
    txt = "".join(c for c in txt if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", txt.lower()).strip()

# Alguns locais conhecidos, associados a estações presentes no modelo.
LOCAIS = {
    "pinacoteca": "Luz",
    "catedral da se": "Sé",
    "museu da lingua portuguesa": "Luz",
    "masp": "Trianon-Masp",
    "hospital das clinicas": "Clínicas",
    "avenida paulista": "Consolação",
    "neo quimica arena": "Corinthians-Itaquera",
    "arena corinthians": "Corinthians-Itaquera",
    "memorial da america latina": "Palmeiras-Barra Funda",
    "shopping metro tatuape": "Tatuapé"
}

ALIAS_ESTACOES = {}
for estacao in GRAFO:
    ALIAS_ESTACOES[normalizar(estacao)] = estacao

def detectar_entidades(texto):
    texto_n = normalizar(texto)
    encontradas = []

    # Locais primeiro, por tamanho decrescente.
    for local in sorted(LOCAIS, key=len, reverse=True):
        pos = texto_n.find(local)
        if pos >= 0:
            encontradas.append((pos, local, LOCAIS[local]))

    # Estações.
    for alias, estacao in sorted(ALIAS_ESTACOES.items(), key=lambda x: len(x[0]), reverse=True):
        pos = texto_n.find(alias)
        if pos >= 0:
            encontradas.append((pos, alias, estacao))

    # Remove duplicatas por estação, mantendo a primeira ocorrência textual.
    encontradas.sort(key=lambda x: (x[0], -len(x[1])))
    unicas = []
    usadas = set()
    for item in encontradas:
        if item[2] not in usadas:
            unicas.append(item)
            usadas.add(item[2])

    return unicas

def interpretar_offline(texto):
    entidades = detectar_entidades(texto)

    if len(entidades) < 2:
        raise ValueError(
            "Não consegui identificar duas estações/locais. "
            "Informe origem e destino usando nomes do modelo."
        )

    origem = entidades[0][2]
    destino = entidades[1][2]

    t = normalizar(texto)
    acessibilidade = any(p in t for p in [
        "acessibilidade", "acessivel", "cadeirante", "cadeira de rodas"
    ])

    algoritmo = "DFS" if "dfs" in t else "BFS"

    bloqueadas = []
    for estacao in GRAFO:
        n = normalizar(estacao)
        padroes = [
            f"{n} fechada", f"{n} fechado",
            f"fechada {n}", f"fechado {n}",
            f"{n} bloqueada", f"{n} bloqueado"
        ]
        if any(p in t for p in padroes):
            bloqueadas.append(estacao)

    elevadores = []
    for estacao in GRAFO:
        n = normalizar(estacao)
        if n in t and ("elevador" in t and ("manutencao" in t or "manutencao" in t)):
            elevadores.append(estacao)

    return {
        "origem": origem,
        "destino": destino,
        "acessibilidade": acessibilidade,
        "bloqueadas": sorted(set(bloqueadas)),
        "elevadores_manutencao": sorted(set(elevadores)),
        "algoritmo": algoritmo
    }

def interpretar_pedido(texto):
    # Modo offline é o padrão e não depende de API.
    return interpretar_offline(texto)

print(json.dumps(
    interpretar_pedido(
        "Estou em Tucuruvi e quero ir para Corinthians-Itaquera, usando BFS."
    ),
    ensure_ascii=False,
    indent=2
))

### Integração opcional com Llama/Groq

A chave não fica escrita no código. Se o aluno possuir uma chave, ela deve ser colocada no ambiente/Secrets do Colab.

O notebook continua funcionando em modo offline caso a API não esteja disponível.

In [ ]:
def prompt_para_llm():
    lista_estacoes = "\n".join(
        f"- {linha}: " + ", ".join(lista)
        for linha, lista in LINHAS.items()
    )
    return f'''
Você é o interpretador do MetrôBot SP 2.0.
Converta o pedido do usuário para JSON.
Use somente esta rede:

{lista_estacoes}

Locais conhecidos:
{json.dumps(LOCAIS, ensure_ascii=False)}

Retorne somente JSON válido com:
origem, destino, acessibilidade, bloqueadas, elevadores_manutencao, algoritmo.

Não invente estações. Se um local não puder ser mapeado, deixe o campo correspondente vazio.
'''

def interpretar_com_llm(texto, api_url=None, modelo=None):
    try:
        import os, requests

        chave = os.environ.get("GROQ_API_KEY")
        if not chave:
            raise RuntimeError("GROQ_API_KEY não configurada.")

        api_url = api_url or "https://api.groq.com/openai/v1/chat/completions"
        modelo = modelo or "llama-3.1-8b-instant"

        payload = {
            "model": modelo,
            "temperature": 0,
            "messages": [
                {"role": "system", "content": prompt_para_llm()},
                {"role": "user", "content": texto}
            ]
        }

        r = requests.post(
            api_url,
            headers={"Authorization": f"Bearer {chave}", "Content-Type": "application/json"},
            json=payload,
            timeout=20
        )
        r.raise_for_status()

        conteudo = r.json()["choices"][0]["message"]["content"].strip()
        conteudo = re.sub(r"^```json\s*", "", conteudo)
        conteudo = re.sub(r"^```\s*", "", conteudo)
        conteudo = re.sub(r"\s*```$", "", conteudo)

        dados = json.loads(conteudo)

        if dados.get("origem") not in GRAFO or dados.get("destino") not in GRAFO:
            raise ValueError("LLM retornou estação fora da rede.")

        dados["bloqueadas"] = [
            e for e in dados.get("bloqueadas", []) if e in GRAFO
        ]
        dados["elevadores_manutencao"] = [
            e for e in dados.get("elevadores_manutencao", []) if e in GRAFO
        ]
        dados["algoritmo"] = "DFS" if str(dados.get("algoritmo")).upper() == "DFS" else "BFS"
        dados["acessibilidade"] = bool(dados.get("acessibilidade", False))

        return dados

    except Exception as exc:
        print("LLM indisponível; usando interpretador offline:", exc)
        return interpretar_offline(texto)

## 7. Narrador da rota

A narrativa abaixo é gerada a partir dos dados efetivamente calculados, evitando inventar estações ou transferências.

In [ ]:
def narrar(resultado):
    if not resultado["rota"]:
        return (
            f"Não foi encontrada uma rota de {resultado['origem']} "
            f"para {resultado['destino']} com as restrições informadas."
        )

    rota = resultado["rota"]
    linhas = resultado["linhas"]
    trans = resultado["transferencias"]

    partes = [
        f"Saída: {resultado['origem']}.",
        f"Destino: {resultado['destino']}.",
        f"Rota com {resultado['paradas']} paradas."
    ]

    if linhas:
        partes.append("Linhas utilizadas: " + " → ".join(dict.fromkeys(linhas)) + ".")

    if trans:
        detalhes = "; ".join(
            f"em {t['estacao']}, trocar de {t['de']} para {t['para']}"
            for t in trans
        )
        partes.append(f"Transferências: {detalhes}.")
    else:
        partes.append("Não é necessária troca de linha.")

    partes.append(
        "Sequência: " + " → ".join(rota) + "."
    )

    if resultado["tempo_estimado_min"] is not None:
        partes.append(
            f"Tempo estimado simplificado: aproximadamente "
            f"{resultado['tempo_estimado_min']} min."
        )

    if resultado["regras"]["alertas_acessibilidade"]:
        partes.append(
            "Atenção: há elevador em manutenção em "
            + ", ".join(resultado["regras"]["alertas_acessibilidade"]) + "."
        )

    return " ".join(partes)

## 8. Mapa visual das 3 linhas

A visualização usa as cores oficiais informadas no desafio e destaca:
- estação da rota;
- estação visitada pela busca;
- estação bloqueada;
- estação de integração.

In [ ]:
def desenhar_mapa(resultado=None):
    rota = set(resultado["rota"] or []) if resultado else set()
    visitados = set(resultado["visitados"] or []) if resultado else set()
    bloqueadas = set(resultado["bloqueadas"] or []) if resultado else set()

    css = '''
    <style>
    .metro-wrap {font-family:Arial,sans-serif; margin:10px 0;}
    .metro-line {margin:12px 0;}
    .metro-title {font-weight:bold; margin-bottom:6px;}
    .stations {display:flex; flex-wrap:wrap; gap:5px; align-items:center;}
    .station {padding:6px 8px; border-radius:12px; color:white;
              font-size:12px; font-weight:600; border:2px solid transparent;}
    .route {box-shadow:0 0 0 3px #ffd54f;}
    .visited {opacity:.55;}
    .blocked {background:#333 !important; text-decoration:line-through;}
    .integration {border-style:dashed;}
    .legend {font-size:12px; margin-top:8px;}
    </style>
    '''

    partes = [css, '<div class="metro-wrap">']
    for linha, lista in LINHAS.items():
        partes.append(
            f'<div class="metro-line"><div class="metro-title">{html.escape(linha)}</div>'
        )
        partes.append('<div class="stations">')

        for estacao in lista:
            classes = ["station"]
            if estacao in rota:
                classes.append("route")
            if estacao in visitados and estacao not in rota:
                classes.append("visited")
            if estacao in bloqueadas:
                classes.append("blocked")
            if estacao in INTEGRACOES:
                classes.append("integration")

            bg = CORES[linha]
            partes.append(
                f'<span class="{" ".join(classes)}" style="background:{bg}">'
                f'{html.escape(estacao)}</span>'
            )

        partes.append("</div></div>")

    partes.append(
        '<div class="legend">Amarelo ao redor = rota | '
        'Opacidade reduzida = visitada | Risca = bloqueada | '
        'Borda tracejada = integração.</div>'
    )
    partes.append("</div>")

    display(HTML("".join(partes)))

## 9. Tabela-verdade

Regra proposicional usada:

**P ∧ Q → A**

Onde:
- P = estação está fechada;
- Q = a rota utiliza a estação;
- A = alerta de desvio.

In [ ]:
def tabela_verdade_regra():
    linhas = []
    for P, Q in product([False, True], repeat=2):
        antecedente = P and Q
        A = antecedente
        linhas.append((P, Q, antecedente, A))

    cab = ["P: fechada", "Q: usada pela rota", "P ∧ Q", "A: alerta"]
    tabela = "<table style='border-collapse:collapse'>"
    tabela += "<tr>" + "".join(
        f"<th style='border:1px solid #999;padding:6px'>{h}</th>" for h in cab
    ) + "</tr>"

    for row in linhas:
        tabela += "<tr>" + "".join(
            f"<td style='border:1px solid #999;padding:6px;text-align:center'>{str(v)}</td>"
            for v in row
        ) + "</tr>"

    tabela += "</table>"
    display(HTML(tabela))
    return linhas

TV = tabela_verdade_regra()
assert len(TV) == 4

## 10. Os 6 casos obrigatórios

O bloco abaixo executa exatamente os cenários pedidos no desafio.

In [ ]:
def mostrar_caso(numero, origem, destino, bloqueadas=None):
    resultado = planejar_rota(
        origem, destino,
        bloqueadas=bloqueadas or [],
        algoritmo="BFS"
    )

    print(f"CASO {numero}")
    print(f"{origem} -> {destino}")
    print("Bloqueadas:", bloqueadas or "nenhuma")

    if resultado["rota"]:
        print("Paradas:", resultado["paradas"])
        print("Baldeações:", len(resultado["transferencias"]))
        print("Onde:", resultado["transferencias_nomes"] or "nenhuma")
        print("Rota:", " -> ".join(resultado["rota"]))
    else:
        print("Resultado: No route")

    print("-" * 80)
    return resultado

caso1 = mostrar_caso(
    1, "Tucuruvi", "Corinthians-Itaquera"
)

caso2 = mostrar_caso(
    2, "Vila Madalena", "Jabaquara"
)

caso3 = mostrar_caso(
    3, "Palmeiras-Barra Funda", "Vila Prudente"
)

caso4 = mostrar_caso(
    4, "Tucuruvi", "Brás", ["Sé"]
)

caso5 = mostrar_caso(
    5, "Vila Madalena", "Jabaquara", ["Paraíso"]
)

caso6 = mostrar_caso(
    6, "Vila Prudente", "Jabaquara", ["Paraíso"]
)

### Por que o caso 5 não tem rota e o caso 6 tem?

- **Caso 5:** Vila Madalena começa na Linha 2-Verde e Jabaquara está na Linha 1-Azul. No modelo simplificado, as duas linhas compartilham os pontos de integração **Paraíso** e **Ana Rosa**. Porém, para sair da Linha 2 e chegar à Linha 1, a rota a partir de Vila Madalena precisa alcançar um desses pontos. Com **Paraíso fechado**, o caminho restante pela Linha 2 até **Ana Rosa** deveria permitir a integração. Portanto, a busca deve verificar o grafo real e encontrar o caminho quando ele existir.

- **Caso 6:** Vila Prudente também está na Linha 2-Verde, mas está do lado oposto da linha. Com Paraíso fechado, é possível chegar diretamente a **Ana Rosa** pela Linha 2 e fazer a integração para a Linha 1.

No código, a explicação final deve ser baseada no caminho efetivamente calculado. O teste abaixo verifica o comportamento exigido pelo enunciado.

> **Observação importante:** no grafo do enunciado, o Caso 5 é deliberadamente usado como cenário de bloqueio. A implementação de testes abaixo mantém os resultados obrigatórios fornecidos pelo desafio.

In [ ]:
# Diagnóstico visual dos casos
for nome, resultado in [
    ("Caso 1", caso1), ("Caso 2", caso2), ("Caso 3", caso3),
    ("Caso 4", caso4), ("Caso 5", caso5), ("Caso 6", caso6)
]:
    print(nome, "->", resultado["paradas"], "paradas;",
          len(resultado["transferencias"]), "transferências")

## 11. Testes automatizados

A função `rodar_testes()` usa `assert` para verificar a estrutura do grafo, integrações, rotas e os casos obrigatórios.

In [ ]:
def rodar_testes():
    total = 0

    # 1. Estrutura
    assert len(GRAFO) == 52
    total += 1

    # 2. Nenhum vizinho duplicado
    assert all(len(v) == len(set(v)) for v in GRAFO.values())
    total += 1

    # 3. Paraíso-Ana Rosa pertence às duas linhas
    assert set(linhas_do_trecho("Paraíso", "Ana Rosa")) == {
        "Linha 1-Azul", "Linha 2-Verde"
    }
    total += 1

    # 4. Integrações inferidas
    assert set(INTEGRACOES) == {"Sé", "Paraíso", "Ana Rosa"}
    total += 1

    # 5. Caso 1
    r = planejar_rota("Tucuruvi", "Corinthians-Itaquera")
    assert r["paradas"] == 22
    assert len(r["transferencias"]) == 1
    assert r["transferencias_nomes"] == ["Sé"]
    total += 1

    # 6. Caso 2
    r = planejar_rota("Vila Madalena", "Jabaquara")
    assert r["paradas"] == 14
    assert len(r["transferencias"]) == 1
    total += 1

    # 7. Caso 3
    r = planejar_rota("Palmeiras-Barra Funda", "Vila Prudente")
    assert r["paradas"] == 16
    assert len(r["transferencias"]) == 2
    total += 1

    # 8. Caso 4
    r = planejar_rota("Tucuruvi", "Brás", ["Sé"])
    assert r["rota"] is None
    total += 1

    # 9. Caso 5
    r = planejar_rota("Vila Madalena", "Jabaquara", ["Paraíso"])
    assert r["rota"] is None
    total += 1

    # 10. Caso 6
    r = planejar_rota("Vila Prudente", "Jabaquara", ["Paraíso"])
    esperado = [
        "Vila Prudente", "Tamanduateí", "Sacomã", "Alto do Ipiranga",
        "Santos-Imigrantes", "Chácara Klabin", "Ana Rosa", "Vila Mariana",
        "Santa Cruz", "Praça da Árvore", "Saúde", "São Judas",
        "Conceição", "Jabaquara"
    ]
    assert r["rota"] == esperado
    assert r["paradas"] == 13
    total += 1

    # 11. Interpretador offline
    p = interpretar_offline(
        "Estou em Tucuruvi e quero ir para Jabaquara usando BFS."
    )
    assert p["origem"] == "Tucuruvi"
    assert p["destino"] == "Jabaquara"
    total += 1

    # 12. Tabela verdade
    assert len(TV) == 4
    total += 1

    print(f"Todos os testes passaram: {total} asserts.")

rodar_testes()

## 12. Interface interativa com ipywidgets

A interface permite:
- escrever uma solicitação em linguagem natural;
- escolher origem e destino;
- escolher BFS ou DFS;
- ativar acessibilidade;
- marcar estações fechadas;
- marcar elevadores em manutenção;
- calcular e visualizar a rota.

In [ ]:
estacao_options = sorted(GRAFO)

entrada_natural = widgets.Textarea(
    value="",
    placeholder="Ex.: Estou em Tucuruvi e quero ir para Corinthians-Itaquera",
    description="Pedido:",
    layout=widgets.Layout(width="100%", height="80px")
)

btn_interpretar = widgets.Button(
    description="Interpretar pedido",
    button_style="info"
)

origem_w = widgets.Dropdown(
    options=estacao_options, description="Origem:", layout=widgets.Layout(width="48%")
)
destino_w = widgets.Dropdown(
    options=estacao_options, description="Destino:", layout=widgets.Layout(width="48%")
)

algoritmo_w = widgets.ToggleButtons(
    options=["BFS", "DFS"],
    value="BFS",
    description="Busca:"
)

acessibilidade_w = widgets.Checkbox(
    value=False, description="Preciso de acessibilidade"
)

bloqueadas_w = widgets.SelectMultiple(
    options=estacao_options,
    description="Fechadas:",
    rows=6,
    layout=widgets.Layout(width="48%")
)

elevadores_w = widgets.SelectMultiple(
    options=estacao_options,
    description="Elevadores em manutenção:",
    rows=6,
    layout=widgets.Layout(width="48%")
)

btn_buscar = widgets.Button(
    description="Calcular rota",
    button_style="success"
)

saida = widgets.Output()

def on_interpretar(_):
    with saida:
        clear_output()
        try:
            dados = interpretar_pedido(entrada_natural.value)
            origem_w.value = dados["origem"]
            destino_w.value = dados["destino"]
            algoritmo_w.value = dados.get("algoritmo", "BFS")
            acessibilidade_w.value = dados.get("acessibilidade", False)
            bloqueadas_w.value = tuple(dados.get("bloqueadas", []))
            elevadores_w.value = tuple(dados.get("elevadores_manutencao", []))

            print("JSON interpretado:")
            print(json.dumps(dados, ensure_ascii=False, indent=2))
        except Exception as e:
            print("Erro:", e)

def on_buscar(_):
    with saida:
        clear_output()
        resultado = planejar_rota(
            origem_w.value,
            destino_w.value,
            bloqueadas=list(bloqueadas_w.value),
            acessibilidade=acessibilidade_w.value,
            elevadores_manutencao=list(elevadores_w.value),
            algoritmo=algoritmo_w.value
        )

        print(narrar(resultado))
        print()
        print("Regras disparadas:")
        for regra in resultado["regras"]["regras_disparadas"]:
            print("-", regra)

        print()
        desenhar_mapa(resultado)

        print()
        print("Estações visitadas pela busca:")
        print(" -> ".join(resultado["visitados"]))

btn_interpretar.on_click(on_interpretar)
btn_buscar.on_click(on_buscar)

display(
    widgets.VBox([
        entrada_natural,
        widgets.HBox([btn_interpretar, btn_buscar]),
        widgets.HBox([origem_w, destino_w]),
        algoritmo_w,
        acessibilidade_w,
        widgets.HBox([bloqueadas_w, elevadores_w]),
        saida
    ])
)

## 13. Exemplo de utilização do mapa

In [ ]:
exemplo = planejar_rota("Vila Prudente", "Jabaquara", ["Paraíso"])
print(narrar(exemplo))
desenhar_mapa(exemplo)

## 14. Bônus — busca que prioriza menos transferências

Uma BFS simples minimiza o número de trechos. Para priorizar menos transferências, o estado precisa considerar também a linha atual:

**estado = (estação, linha_atual)**

Isso evita tratar simplesmente a estação como estado quando a mesma estação pode ser usada por mais de uma linha.

In [ ]:
def busca_menos_transferencias(origem, destino, bloqueadas=None):
    bloqueadas = set(bloqueadas or [])

    if origem in bloqueadas or destino in bloqueadas:
        return None

    estados_iniciais = [(origem, linha) for linha in linhas_da_estacao(origem)]

    fila = deque(estados_iniciais)
    anterior = {estado: None for estado in estados_iniciais}

    while fila:
        estacao, linha_atual = fila.popleft()

        if estacao == destino:
            estado_final = (estacao, linha_atual)
            caminho_estados = []
            cur = estado_final
            while cur is not None:
                caminho_estados.append(cur)
                cur = anterior[cur]
            caminho_estados.reverse()
            return caminho_estados

        # Movimentação pela mesma linha
        if linha_atual in LINHAS:
            lista = LINHAS[linha_atual]
            idx = lista.index(estacao)
            for j in [idx - 1, idx + 1]:
                if 0 <= j < len(lista):
                    prox = lista[j]
                    if prox in bloqueadas:
                        continue
                    estado = (prox, linha_atual)
                    if estado not in anterior:
                        anterior[estado] = (estacao, linha_atual)
                        fila.append(estado)

        # Transferência na estação
        for nova_linha in linhas_da_estacao(estacao):
            if nova_linha != linha_atual:
                estado = (estacao, nova_linha)
                if estado not in anterior:
                    anterior[estado] = (estacao, linha_atual)
                    fila.append(estado)

    return None

bonus = busca_menos_transferencias("Palmeiras-Barra Funda", "Vila Prudente")
print("Estados encontrados:", len(bonus) if bonus else None)
print(bonus)

## 15. Declaração de uso de IA

**Declaração sugerida para o trabalho:**

> Foi utilizada Inteligência Artificial como ferramenta de apoio à elaboração e revisão da solução, especialmente na organização do código, estruturação das regras, implementação das buscas e preparação dos testes. A solução foi revisada e testada no ambiente Python, mantendo o entendimento dos componentes e das regras implementadas.

## 16. Checklist final

- [x] 3 linhas modeladas.
- [x] 52 estações distintas.
- [x] Estações compartilhadas como um único nó.
- [x] Linhas armazenadas por trecho.
- [x] R6 deduzida por inferência.
- [x] R7 adicionada.
- [x] BFS.
- [x] DFS.
- [x] Interpretador offline.
- [x] Estrutura JSON.
- [x] Narrador.
- [x] Interface ipywidgets.
- [x] Mapa visual.
- [x] Tabela-verdade.
- [x] 6 casos obrigatórios.
- [x] Testes com mais de 8 `assert`s.
- [x] Declaração de uso de IA.